In [70]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("../data/processed/clean.csv")
print(df.shape)

(197283, 20)


In [4]:
# free drivers = working drivers minus busy drivers
df["available_dashers"] = df["total_onshift_dashers"] - df["total_busy_dashers"]

# share of drivers that are busy (+1 avoids dividing by zero)
df["busy_ratio"] = df["total_busy_dashers"] / (df["total_onshift_dashers"] + 1)

# waiting orders for each driver
df["orders_per_dasher"] = df["total_outstanding_orders"] / (df["total_onshift_dashers"] + 1)

# average price of one item
df["avg_item_price"] = df["subtotal"] / df["total_items"].clip(lower=1)

# 1 if Saturday or Sunday, else 0
df["is_weekend"] = (df["weekday"] >= 5).astype(int)

df[["available_dashers", "busy_ratio", "orders_per_dasher", "avg_item_price", "is_weekend"]].describe()

,available_dashers,busy_ratio,orders_per_dasher,avg_item_price,is_weekend
count,197283.000000,197283.000000,197283.000000,197283.000000,197283.000000
mean,3.063270,0.887676,1.137185,975.281695,0.345286
std,10.949887,0.367879,0.437927,517.010948,0.475463
min,-64.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.791667,0.900000,647.666667,0.000000
50%,2.000000,0.900000,1.100000,895.000000,0.000000
75%,5.000000,0.976744,1.387097,1195.000000,1.000000
max,86.000000,34.000000,23.500000,14700.000000,1.000000


In [5]:
# sort by order time
df = df.sort_values("created_at").reset_index(drop=True)

# first 80% of rows = train, last 20% = test
split = int(len(df) * 0.8)

print("train ends:", df["created_at"][split - 1])
print("test starts:", df["created_at"][split])

train ends: 2015-02-13 04:06:37
test starts: 2015-02-13 04:06:44


In [6]:
# remove columns the model must not see (they would leak the answer)
X = df.drop(columns=["created_at", "actual_delivery_time", "duration_min", "store_id"])
y = df["duration_min"]

# turn category columns into 0/1 columns
X = pd.get_dummies(X, columns=["market_id", "order_protocol", "store_primary_category"], dtype=int)

# train = first part, test = last part
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(X_train.shape, X_test.shape)

(157826, 108) (39457, 108)


In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

results = []

def evaluate(name, pred):
    mae = mean_absolute_error(y_test, pred)   
    r2 = r2_score(y_test, pred)               
    results.append([name, round(mae, 2), round(r2, 3)])
    print(name, "| MAE:", round(mae, 2), "| R2:", round(r2, 3))

# baseline: always guess the median delivery time of the train data
baseline_pred = [y_train.median()] * len(y_test)
evaluate("Median baseline", baseline_pred)

Median baseline | MAE: 14.0 | R2: -0.087


In [11]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# copy the data and cap the extreme values (for Ridge only)
X_train_r, X_test_r = X_train.copy(), X_test.copy()
for col in ["available_dashers", "busy_ratio", "orders_per_dasher", "avg_item_price"]:
    low, high = X_train[col].quantile(0.01), X_train[col].quantile(0.99)
    X_train_r[col] = X_train_r[col].clip(low, high)
    X_test_r[col] = X_test_r[col].clip(low, high)

# scale the numbers, then fit a Ridge model
ridge = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
ridge.fit(X_train_r, y_train)
evaluate("Ridge", ridge.predict(X_test_r))

Ridge | MAE: 11.62 | R2: 0.276


In [12]:
from sklearn.ensemble import RandomForestRegressor

# limited depth keeps it fast; n_jobs=-1 uses all CPU cores
rf = RandomForestRegressor(n_estimators=100, max_depth=12,
                           min_samples_leaf=20, n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)
evaluate("Random Forest", rf.predict(X_test))

Random Forest | MAE: 11.26 | R2: 0.293


In [65]:
import lightgbm as lgb

lgbm = lgb.LGBMRegressor(n_estimators=1000, learning_rate=0.05,
                         num_leaves=31, random_state=42, verbose=-1)
lgbm.fit(X_train, y_train)
evaluate("LightGBM", lgbm.predict(X_test))

LightGBM | MAE: 10.88 | R2: 0.336


In [72]:
import joblib

# 1. keep only the original features (drop experiment columns if present)
extra = ["store_avg", "recent_avg", "orders_last_30min", "demand_per_free_dasher",
         "hour_sin", "hour_cos", "fixed_time_min"]
X_train_f = X_train.drop(columns=[c for c in extra if c in X_train.columns])
X_test_f = X_test[X_train_f.columns]

# 2. train the final model
final_model = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05,
                                num_leaves=31, random_state=42, verbose=-1)
final_model.fit(X_train_f, y_train)

# 3. check the score
pred = final_model.predict(X_test_f)
print("MAE:", round(mean_absolute_error(y_test, pred), 2))
print("R2:", round(r2_score(y_test, pred), 3))

# 4. save the model and the column order
joblib.dump(final_model, "../models/lgbm_eta.pkl")
joblib.dump(list(X_train_f.columns), "../models/feature_columns.pkl")

# 5. test that the saved model loads and gives the same predictions
loaded = joblib.load("../models/lgbm_eta.pkl")
cols = joblib.load("../models/feature_columns.pkl")
same = np.allclose(loaded.predict(X_test_f[cols]), pred)
print("saved model works:", same)

MAE: 10.89
R2: 0.336
saved model works: True
